# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading, inspecting, and analyzing the FAIR\^2 dataset (mass spectrometry analysis of extracellular vesicles) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset follows the [Croissant standard](https://mlcommons.org/croissant/) and is accessible via a public Croissant schema URL.

In [ ]:
# Install required library
!pip install mlcroissant

## 1. Data Loading

We start by loading the dataset metadata and inspecting its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Let's explore the available Record Sets in the dataset. We'll list all record sets with their display names and `@id` values. Each record set has a collection of fields (columns), and we will enumerate these as well using their `@id`s.

In [ ]:
# List all record sets and their fields by @id

record_sets = dataset.metadata.recordSets
for record_set in record_sets:
    print(f"Record Set Name: {getattr(record_set, 'name', 'N/A')}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    fields = getattr(record_set, 'fields', [])
    for field in fields:
        print(f"    - {getattr(field, 'name', 'N/A')}  (@id: {field.id})  [type: {getattr(field, 'dataType', 'N/A')}]" )
    print()

## 3. Data Extraction

We can now extract data from a record set. Use the `@id` of the record set (as displayed above) to reference it. Here, we will load all records from a chosen record set into a Pandas DataFrame for analysis.

> **Note:** The main tabular record set typically has a descriptive `@id` like `cr:RS_Proteins` or similar, but please reference the actual `@id` found in the previous step.

In [ ]:
# Choose one or more record set @id(s) from the overview above

# Example: suppose the main data table record set is 'cr:RS_Proteins'
# Replace with the actual @id from your dataset
main_record_set_id = dataset.metadata.recordSets[0].id  # Use the first as an example

dataframes = {}

# In general, you might want to loop through all record sets:
for record_set in dataset.metadata.recordSets:
    records = list(dataset.records(record_set=record_set.id))
    df = pd.DataFrame(records)
    dataframes[record_set.id] = df
    print(f"Loaded {len(df)} records for record set {record_set.id}")

# Display columns of the main dataframe
main_df = dataframes[main_record_set_id]
print("Columns:\n", main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's process and explore numeric fields. We'll pick a numeric field (e.g., counts, molecular weight) using its `@id` (as listed in the Record Set overview; e.g. `cr:F_MW` or similar). We'll show basic filtering, normalization, and grouping.

*Be sure to update the field `@id` variables used below with the actual field id from your dataset after reviewing the columns list above!*

In [ ]:
# Replace with a numeric field @id as revealed above (e.g. 'cr:F_MW', 'cr:F_peptide_count')
# If the loaded DataFrame columns are not @id but names, adjust accordingly
# For demonstration, we'll use the first float/int column we find

numeric_field_id = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    raise Exception("No numeric field found in the main dataframe.")
print(f"Using numeric field for analysis: {numeric_field_id}")

threshold = main_df[numeric_field_id].mean()
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records ({len(filtered_df)} samples) with {numeric_field_id} > mean ({threshold:.2f}):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

# Try grouping by a categorical/descriptor field. Pick the first object (string) type field.
group_field_id = None
for col in main_df.columns:
    if pd.api.types.is_string_dtype(main_df[col]) and col != numeric_field_id:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field (as a histogram), and, if grouping is available, show the means per group.

> Note: This requires matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Grouped means plot
if group_field_id:
    plt.figure(figsize=(10,4))
    order = grouped_df.sort_values(numeric_field_id)[group_field_id].values
    sns.barplot(
        data=grouped_df,
        x=group_field_id, y=numeric_field_id, order=order
    )
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook we:
- Loaded FAIR\^2 mass spectrometry dataset metadata and tabular records using `mlcroissant`.
- Inspected available Record Sets and their fields (with full `@id` referencing for reproducibility).
- Extracted data to Pandas DataFrames, performed basic filtering and normalization, and visualized key distributions.

You may further extend this notebook for downstream tasks such as:
- Custom data preprocessing (outlier removal, advanced imputation)
- Integration with protein annotation databases
- Building machine learning models for protein property prediction

For more information, see [Croissant specification](https://mlcommons.org/croissant/) and the [mlcroissant library documentation](https://github.com/mlcommons/croissant).